<a href="https://colab.research.google.com/github/sanadv/Classical_Machine_Learning/blob/main/Classical_Machine_Leaning_Chapter_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np

# Input sentence
sentence = "hello how are you"
chars = list(set(sentence))
print(chars)
vocab_size = len(chars)
char_to_idx = {char: i for i, char in enumerate(chars)}
print(char_to_idx)
idx_to_char = {i: char for i, char in enumerate(chars)}
print(idx_to_char)

# Hyperparameters
hidden_size = 4  # Size of hidden state
sequence_size = 5  # Sequence length
learning_rate = 0.01

# One-hot encoding
def one_hot(index, size):
    vec = np.zeros(size)
    vec[index] = 1
    return vec

# Initialize parameters
global Wxh, Whh, Why, bh, by
Wxh = np.round( np.random.randn(hidden_size, vocab_size) * 0.1,3)
Whh = np.round( np.random.randn(hidden_size, hidden_size) * 0.1,3)
Why = np.round( np.random.randn(vocab_size, hidden_size) * 0.1,3)


bh = np.zeros((hidden_size, 1))
by = np.zeros((vocab_size, 1))

# Activation function (softmax)
def softmax(x):
    exp_x = np.exp(x - np.max(x))
    return exp_x / exp_x.sum(axis=0)

# Forward and Backward pass
def rnn_forward_backward(sentence):
    global Wxh, Whh, Why, bh, by
    inputs = [one_hot(char_to_idx[char], vocab_size) for char in sentence]
    targets = inputs[1:] + [inputs[0]]

    loss = 0
    for i in range(0, len(inputs) - sequence_size + 1, sequence_size):
        hs = [np.zeros((hidden_size, 1))]
        ys = []
        ps = []

        # Forward pass
        for t in range(sequence_size):

            x = inputs[i + t].reshape(vocab_size, 1)
            ht = np.tanh(np.dot(Wxh, x) + np.dot(Whh, hs[-1]) + bh)
            yt = np.dot(Why, ht) + by
            pt = softmax(yt)
            loss += -np.log(pt[np.argmax(targets[i + t])])
            hs.append(ht)
            ys.append(yt)
            ps.append(pt)

        # Backward pass
        dWxh, dWhh, dWhy = np.zeros_like(Wxh), np.zeros_like(Whh), np.zeros_like(Why)
        dbh, dby = np.zeros_like(bh), np.zeros_like(by)
        dh_next = np.zeros_like(hs[0])
        for t in reversed(range(sequence_size)):
            dy = ps[t]
            dy[np.argmax(targets[i + t])] -= 1
            dWhy += np.dot(dy, hs[t+1].T)
            dby += dy
            dh = np.dot(Why.T, dy) + dh_next
            dh_raw = (1 - hs[t+1] ** 2) * dh
            dbh += dh_raw
            dWxh += np.dot(dh_raw, inputs[i + t].reshape(1, -1))
            dWhh += np.dot(dh_raw, hs[t].T)
            dh_next = np.dot(Whh.T, dh_raw)

        # Gradient Clipping
        for dparam in [dWxh, dWhh, dWhy, dbh, dby]:
            np.clip(dparam, -5, 5, out=dparam)

        # Update parameters
        Wxh -= learning_rate * dWxh
        Whh -= learning_rate * dWhh
        Why -= learning_rate * dWhy
        bh -= learning_rate * dbh
        by -= learning_rate * dby

    print(f"Loss: {loss}")
    print("Gradients:")
    print("dWxh:", dWxh)
    print("dWhh:", dWhh)
    print("dWhy:", dWhy)
    print("dbh:", dbh)
    print("dby:", dby)
# Run RNN
rnn_forward_backward(sentence)
